### Load Multiple-Choice Questions

In [1]:
import json

with open("data/ptbr-multiple-choice-qa-pairs.json", "r", encoding="utf-8") as f:
    mcq_items = json.load(f)

In [2]:
import re
from typing import Dict, List

def expand_references_in_question(
	question: str,
	images: Dict[str, str] | None,
	tables: Dict[str, str] | None,
) -> str:
	images = images or {}
	tables = tables or {}

	# Find tokens in parentheses like (image1), (table1), etc.
	tokens = re.findall(r"\(([^)]+)\)", question or "")
	# Preserve first-seen order without duplicates
	seen: set[str] = set()
	references: List[str] = []
	for t in tokens:
		if t in seen:
			continue
		if t in images or t in tables:
			seen.add(t)
			references.append(t)

	if not references:
		return question

	parts = [question, "\n\nReferenced content:"]
	for key in references:
		parts.append(f"[{key}]\n{images.get(key) if key in images else tables.get(key, '')}")
	return "\n".join(parts)

In [3]:
def build_prompt(question_text: str, options_text: Dict[str, str]) -> str:
	header = (
		"Solve the following math multiple-choice question. Make sure to put the correct option letter (A,B,C,D or E), and only the correct option letter, inside \\boxed{}.\n\n"
	)
	# Ensure options appear ordered as A..E (only those present)
	ordered_keys = [k for k in ["A", "B", "C", "D", "E"] if k in options_text]
	options_lines = "\n".join(f"{k}) {options_text[k]}" for k in ordered_keys)
	prompt = f"{header}Question: {question_text}\n\n{options_lines}\n"
	return prompt

In [4]:
prompts = []

for item in mcq_items:
    if not item.get("multiple_choice_question", False):
        continue
    question = item.get("question_verbatim")
    options = item.get("options_verbatim")
    if not question or not options:
        print(f"Skipping item without valid question/options: {item.get('id', 'unknown id')}")
        continue
    expanded_question = expand_references_in_question(
			question,
			item.get("images"),
			item.get("tables"),
	)
    prompt_text = build_prompt(expanded_question, options)
    prompts.append({
        "id": item.get("id"),
        "prompt": prompt_text,
    })

In [5]:
for entry in prompts:
    # Pretty-print with id context
    pid = entry.get("id", "unknown")
    print(f"id={pid}\n{entry.get('prompt','')}\n")

id=1133
Solve the following math multiple-choice question. Make sure to put the correct option letter (A,B,C,D or E), and only the correct option letter, inside \boxed{}.

Question: O pai de Carolina mediu o comprimento da mesa da sala com sua mão e contou $8$ palmos. Ela também mediu a mesa do mesmo modo e contou $11$ palmos. Qual é o tamanho do palmo de Carolina, se o palmo de seu pai mede $22$ centímetros?

A) 16 cm
B) 14 cm
C) 13 cm
D) 12 cm
E) 11 cm


id=1134
Solve the following math multiple-choice question. Make sure to put the correct option letter (A,B,C,D or E), and only the correct option letter, inside \boxed{}.

Question: Quantos sinais de adição foram utilizados na expressão $2+0+1+3+2+0+1+3+2+0+1+3+\ldots+2+0+1=2013$?

A) 503
B) 1342
C) 2012
D) 2013
E) 2016


id=1135
Solve the following math multiple-choice question. Make sure to put the correct option letter (A,B,C,D or E), and only the correct option letter, inside \boxed{}.

Question: Marcos fez cinco provas de Matemá

In [6]:
with open("prompts/ptbr-prompts.json", "w", encoding="utf-8") as f:
    json.dump(prompts, f, ensure_ascii=False, indent=2)